# 01 — EDA & Limpeza

> **Etapa 2 do Datathon FIAP Fase 5 — Associação Passos Mágicos**

## Objetivos

1. **Inventariar** a estrutura real do dataset (3 abas: PEDE2022, PEDE2023, PEDE2024)
2. **Diagnosticar** problemas de qualidade (missing, outliers, tipos)
3. **Normalizar** o schema inconsistente entre anos
4. **Consolidar** em formato *long* (1 linha por aluno-ano)
5. **Visualizar** distribuições, correlações e evolução temporal
6. **Investigar** padrões anômalos (IPS 2023, queda de IEG em 2024)
7. **Exportar** `data/processed/alunos_long.parquet` pronto pra modelagem

## Decisões já aprovadas (relatório 2A)

| # | Decisão |
|---|---|
| 1 | Nomes canônicos em `snake_case` |
| 2 | Drop de colunas 100% vazias (Cg, Cf, Ct, Rec Av*, Indicado, Atingiu PV, Destaque*) |
| 3 | Fase normalizada pra `int` 0-9 (Alfa=0, …, Universitários=9) |
| 4 | Idade 2023 com bug datetime corrigida via `.day` |
| 5 | "INCLUIR" em Pedra 2024 → `NaN` (38 universitários ingressantes) |
| 6 | IPS 2023 anômalo → mantido como está, investigar depois |
| 7 | Componente DL: corpus PEDE (B1) + LSTM sequencial (backup) |
| 8 | Inglês (64% missing estrutural) → feature `tem_nota_ingles` |
| 9 | Target forward-looking com regra composta (OU)

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_RAW / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

assert XLSX_PATH.exists(), f"Arquivo não encontrado em {XLSX_PATH}"
print(f"✅ Dataset encontrado: {XLSX_PATH.name}")
print(f"   Tamanho: {XLSX_PATH.stat().st_size / 1024:.1f} KB")

## 2. Carregamento das 3 abas

Cada aba corresponde a um ano do PEDE (Pesquisa Extensiva do Desenvolvimento Educacional). Carregamos tudo *raw* primeiro pra fazer o inventário antes de tocar em qualquer coluna.

In [ ]:
xl = pd.ExcelFile(XLSX_PATH)
print(f"Abas encontradas: {xl.sheet_names}")

df22_raw = pd.read_excel(XLSX_PATH, sheet_name="PEDE2022")
df23_raw = pd.read_excel(XLSX_PATH, sheet_name="PEDE2023")
df24_raw = pd.read_excel(XLSX_PATH, sheet_name="PEDE2024")

print(f"\nPEDE2022: {df22_raw.shape[0]:4d} linhas × {df22_raw.shape[1]:2d} colunas")
print(f"PEDE2023: {df23_raw.shape[0]:4d} linhas × {df23_raw.shape[1]:2d} colunas")
print(f"PEDE2024: {df24_raw.shape[0]:4d} linhas × {df24_raw.shape[1]:2d} colunas")
print(f"\nTotal bruto: {len(df22_raw) + len(df23_raw) + len(df24_raw)} linhas")

## 3. Inventário inicial: colunas, tipos e missing

Antes de limpar, precisamos ver **o que tem** em cada aba. O schema **muda entre anos** — isso é uma das principais dores deste dataset.

In [ ]:
def inventariar(df, nome):
    """Gera um inventário enxuto de um DataFrame."""
    inv = pd.DataFrame({
        "coluna": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().sum().values / len(df) * 100).round(1),
        "n_unicos": [df[c].nunique(dropna=True) for c in df.columns],
    })
    print(f"\n{'='*60}")
    print(f"INVENTÁRIO — {nome}  (shape: {df.shape})")
    print(f"{'='*60}")
    return inv


inv22 = inventariar(df22_raw, "PEDE2022")
inv22

In [ ]:
inv23 = inventariar(df23_raw, "PEDE2023")
inv23

In [ ]:
inv24 = inventariar(df24_raw, "PEDE2024")
inv24

### 3.1 Colunas 100% vazias — candidatas a descarte

Muitas colunas em 2023 e 2024 estão **completamente vazias**. São campos legados (rankings Cg/Cf/Ct, recomendações da comissão, destaques) que só foram preenchidos em 2022 e depois descontinuados pela Passos.

In [ ]:
def colunas_totalmente_vazias(df, nome):
    vazias = df.columns[df.isna().all()].tolist()
    print(f"{nome}: {len(vazias)} colunas 100% vazias")
    for c in vazias:
        print(f"  - {c}")
    return vazias


v22 = colunas_totalmente_vazias(df22_raw, "PEDE2022")
v23 = colunas_totalmente_vazias(df23_raw, "PEDE2023")
v24 = colunas_totalmente_vazias(df24_raw, "PEDE2024")

**Achado:** 2022 não tem colunas 100% vazias, mas **2023 e 2024 têm várias** — são exatamente os campos que foram descontinuados. A decisão #2 do relatório 2A é dropar tudo isso no schema consolidado.

### 3.2 Presença dos 7 indicadores por ano

Os 7 indicadores do PEDE (IAA, IEG, IPS, IPP, IDA, IPV, IAN) são a matéria-prima do projeto. Vamos confirmar que eles existem em todos os anos.

In [ ]:
indicadores = ["IAA", "IEG", "IPS", "IPP", "IDA", "IPV", "IAN"]

presenca = pd.DataFrame({
    "2022": [ind in df22_raw.columns for ind in indicadores],
    "2023": [ind in df23_raw.columns for ind in indicadores],
    "2024": [ind in df24_raw.columns for ind in indicadores],
}, index=indicadores)

presenca.replace({True: "✅", False: "❌"})

> ⚠️ **IPP não existe em PEDE2022.** Todas as features temporais que dependem de IPP só podem usar 2023→2024. Isso é uma limitação estrutural que vamos contornar no feature engineering.

## 4. Funções de limpeza e normalização

Agora que sabemos **onde estão os problemas**, construímos funções reutilizáveis que normalizam cada aba pro schema canônico.

### 4.1 Normalização da Fase

A coluna `Fase` tem **3 formatos diferentes** entre os anos:

| ano | formato | exemplo |
|---|---|---|
| 2022 | `int` direto | `0`, `1`, …, `7` |
| 2023 | string `"FASE N"` | `"ALFA"`, `"FASE 1"`, …, `"FASE 8"` |
| 2024 | código de turma | `"ALFA"`, `"1A"`, `"1B"`, …, `"8F"`, `9` |

A função abaixo converte tudo pra `int` canônico (0 = Alfa, 1-8 = fases regulares, 9 = universitários pós-fase-8).

In [ ]:
def normalizar_fase(valor):
    """Converte diferentes formatos de Fase pra int canônico 0-9."""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return int(valor)
    v = str(valor).strip().upper()
    if v == "ALFA":
        return 0
    m = re.match(r"FASE\s*(\d+)", v)
    if m:
        return int(m.group(1))
    # Código de turma tipo "1A", "8F" — pega o número inicial
    m = re.match(r"(\d+)", v)
    if m:
        return int(m.group(1))
    return np.nan


def extrair_turma(valor):
    """Extrai sufixo alfabético ('1A' → 'A'). Retorna NaN se não houver."""
    if pd.isna(valor) or isinstance(valor, (int, float)):
        return np.nan
    v = str(valor).strip().upper()
    m = re.match(r"\d+([A-Z]+)", v)
    return m.group(1) if m else np.nan


# Testes rápidos
testes = [
    (0, 0), (1, 1), (7, 7), (9, 9),
    ("ALFA", 0), ("FASE 1", 1), ("Fase 8", 8),
    ("1A", 1), ("2B", 2), ("8F", 8),
    (np.nan, np.nan),
]
for entrada, esperado in testes:
    resultado = normalizar_fase(entrada)
    ok = "✅" if (resultado == esperado) or (pd.isna(resultado) and pd.isna(esperado)) else "❌"
    print(f"  {ok} normalizar_fase({entrada!r}) = {resultado!r} (esperado {esperado!r})")

### 4.2 Correção do bug datetime na Idade 2023

Em `PEDE2023`, a coluna `Idade` foi lida pelo Excel de forma torta: idades como `8` viraram `datetime(1900, 1, 8)`. Isso é o bug clássico do Excel interpretando inteiros como datas seriais.

A solução é extrair o `.day` dos valores que vieram como datetime.

In [ ]:
def corrigir_idade_datetime(valor):
    """Idade 8 → datetime(1900,1,8) → extrai .day = 8."""
    if pd.isna(valor):
        return np.nan
    if hasattr(valor, "day"):
        try:
            return int(valor.day)
        except (AttributeError, TypeError):
            pass
    try:
        return int(valor)
    except (ValueError, TypeError):
        return np.nan


# Mostrar amostra de idades problemáticas antes e depois
amostra_idade = df23_raw["Idade"].head(15).tolist()
print("Antes (raw, amostra):")
for v in amostra_idade:
    print(f"  {repr(v)}")
print("\nDepois (corrigido):")
for v in amostra_idade:
    print(f"  {corrigir_idade_datetime(v)}")

### 4.3 Normalização da Pedra

- Unifica grafia (`Agata` → `Ágata`)
- `"INCLUIR"` em 2024 → `NaN` (38 universitários ingressantes sem avaliação completa)

In [ ]:
def normalizar_pedra(valor):
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip()
    if v.upper() == "INCLUIR" or v == "":
        return np.nan
    mapa = {
        "Agata": "Ágata", "ÁGATA": "Ágata", "ágata": "Ágata",
        "QUARTZO": "Quartzo", "quartzo": "Quartzo",
        "AMETISTA": "Ametista", "ametista": "Ametista",
        "TOPÁZIO": "Topázio", "TOPAZIO": "Topázio", "Topazio": "Topázio",
    }
    return mapa.get(v, v)

### 4.4 Pipelines por ano

Uma função de limpeza por aba. Cada uma sabe lidar com as idiossincrasias do seu ano (nomes de coluna, bugs específicos) e retorna um DataFrame no schema canônico.

In [ ]:
def limpar_2022(df):
    """Limpa e padroniza PEDE2022."""
    out = pd.DataFrame()
    out["ra"] = df["RA"].astype(str)
    out["ano"] = 2022
    out["fase"] = df["Fase"].apply(normalizar_fase)
    out["turma"] = df["Turma"].astype(str)
    out["nome_anon"] = df["Nome"].astype(str)
    out["ano_nascimento"] = df["Ano nasc"]
    out["idade"] = pd.to_numeric(df["Idade 22"], errors="coerce")
    out["genero"] = df["Gênero"].astype(str)
    out["ano_ingresso"] = pd.to_numeric(df["Ano ingresso"], errors="coerce")
    out["instituicao_ensino"] = df["Instituição de ensino"].astype(str)
    out["escola"] = np.nan
    out["pedra"] = df["Pedra 22"].apply(normalizar_pedra)
    out["inde"] = pd.to_numeric(df["INDE 22"], errors="coerce")
    out["ian"] = pd.to_numeric(df["IAN"], errors="coerce")
    out["ida"] = pd.to_numeric(df["IDA"], errors="coerce")
    out["ieg"] = pd.to_numeric(df["IEG"], errors="coerce")
    out["iaa"] = pd.to_numeric(df["IAA"], errors="coerce")
    out["ips"] = pd.to_numeric(df["IPS"], errors="coerce")
    out["ipp"] = np.nan  # ❗ IPP não existe em 2022
    out["ipv"] = pd.to_numeric(df["IPV"], errors="coerce")
    out["nota_mat"] = pd.to_numeric(df["Matem"], errors="coerce")
    out["nota_port"] = pd.to_numeric(df["Portug"], errors="coerce")
    out["nota_ing"] = pd.to_numeric(df["Inglês"], errors="coerce")
    out["fase_ideal"] = df["Fase ideal"].apply(normalizar_fase)
    out["defasagem"] = pd.to_numeric(df["Defas"], errors="coerce")
    return out


def limpar_2023(df):
    """Limpa e padroniza PEDE2023."""
    out = pd.DataFrame()
    out["ra"] = df["RA"].astype(str)
    out["ano"] = 2023
    out["fase"] = df["Fase"].apply(normalizar_fase)
    out["turma"] = df["Turma"].astype(str)
    out["nome_anon"] = df["Nome Anonimizado"].astype(str)
    out["ano_nascimento"] = pd.to_datetime(df["Data de Nasc"], errors="coerce").dt.year
    out["idade"] = df["Idade"].apply(corrigir_idade_datetime)
    out["genero"] = df["Gênero"].astype(str)
    out["ano_ingresso"] = pd.to_numeric(df["Ano ingresso"], errors="coerce")
    out["instituicao_ensino"] = df["Instituição de ensino"].astype(str)
    out["escola"] = np.nan
    out["pedra"] = df["Pedra 2023"].apply(normalizar_pedra)
    out["inde"] = pd.to_numeric(df["INDE 2023"], errors="coerce") if "INDE 2023" in df.columns else np.nan
    out["ian"] = pd.to_numeric(df["IAN"], errors="coerce")
    out["ida"] = pd.to_numeric(df["IDA"], errors="coerce")
    out["ieg"] = pd.to_numeric(df["IEG"], errors="coerce")
    out["iaa"] = pd.to_numeric(df["IAA"], errors="coerce")
    out["ips"] = pd.to_numeric(df["IPS"], errors="coerce")
    out["ipp"] = pd.to_numeric(df["IPP"], errors="coerce")
    out["ipv"] = pd.to_numeric(df["IPV"], errors="coerce")
    out["nota_mat"] = pd.to_numeric(df["Mat"], errors="coerce")
    out["nota_port"] = pd.to_numeric(df["Por"], errors="coerce")
    out["nota_ing"] = pd.to_numeric(df["Ing"], errors="coerce")
    out["fase_ideal"] = df["Fase Ideal"].apply(normalizar_fase)
    out["defasagem"] = pd.to_numeric(df["Defasagem"], errors="coerce")
    return out


def limpar_2024(df):
    """Limpa e padroniza PEDE2024."""
    out = pd.DataFrame()
    out["ra"] = df["RA"].astype(str)
    out["ano"] = 2024
    out["fase"] = df["Fase"].apply(normalizar_fase)
    # ⚠️ Em 2024 a coluna "Fase" contém código de turma ("1A", "2B"),
    # então a turma sai DAÍ, não da coluna "Turma" (que tá vazia/bagunçada).
    out["turma"] = df["Fase"].apply(extrair_turma)
    out["nome_anon"] = df["Nome Anonimizado"].astype(str)
    out["ano_nascimento"] = pd.to_datetime(df["Data de Nasc"], errors="coerce").dt.year
    out["idade"] = pd.to_numeric(df["Idade"], errors="coerce")
    out["genero"] = df["Gênero"].astype(str)
    out["ano_ingresso"] = pd.to_numeric(df["Ano ingresso"], errors="coerce")
    out["instituicao_ensino"] = df["Instituição de ensino"].astype(str)
    out["escola"] = df["Escola"].astype(str) if "Escola" in df.columns else np.nan
    out["pedra"] = df["Pedra 2024"].apply(normalizar_pedra)
    out["inde"] = pd.to_numeric(df["INDE 2024"], errors="coerce")
    out["ian"] = pd.to_numeric(df["IAN"], errors="coerce")
    out["ida"] = pd.to_numeric(df["IDA"], errors="coerce")
    out["ieg"] = pd.to_numeric(df["IEG"], errors="coerce")
    out["iaa"] = pd.to_numeric(df["IAA"], errors="coerce")
    out["ips"] = pd.to_numeric(df["IPS"], errors="coerce")
    out["ipp"] = pd.to_numeric(df["IPP"], errors="coerce")
    out["ipv"] = pd.to_numeric(df["IPV"], errors="coerce")
    out["nota_mat"] = pd.to_numeric(df["Mat"], errors="coerce")
    out["nota_port"] = pd.to_numeric(df["Por"], errors="coerce")
    out["nota_ing"] = pd.to_numeric(df["Ing"], errors="coerce")
    out["fase_ideal"] = df["Fase Ideal"].apply(normalizar_fase)
    out["defasagem"] = pd.to_numeric(df["Defasagem"], errors="coerce")
    return out


print("✅ Funções de limpeza definidas")

## 5. Consolidação em formato *long*

Agora aplicamos as funções e empilhamos em um único DataFrame. Resultado: **1 linha = 1 aluno × 1 ano**, schema único, pronto pra tudo que vier depois.

In [ ]:
df22 = limpar_2022(df22_raw)
df23 = limpar_2023(df23_raw)
df24 = limpar_2024(df24_raw)

alunos = pd.concat([df22, df23, df24], ignore_index=True)

# Features derivadas
alunos["tem_nota_ingles"] = alunos["nota_ing"].notna()
alunos["anos_no_programa"] = alunos["ano"] - alunos["ano_ingresso"]

# Reordenar
colunas_ordem = [
    "ra", "ano", "fase", "turma", "nome_anon",
    "ano_nascimento", "idade", "genero",
    "ano_ingresso", "anos_no_programa",
    "instituicao_ensino", "escola",
    "pedra", "inde",
    "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv",
    "nota_mat", "nota_port", "nota_ing", "tem_nota_ingles",
    "fase_ideal", "defasagem",
]
alunos = alunos[colunas_ordem]

print(f"✅ Dataset consolidado: {alunos.shape}")
alunos.head()

## 6. Validações pós-limpeza

In [ ]:
# 6.1 — Chave (ra, ano) deve ser única
dups = alunos.duplicated(subset=["ra", "ano"]).sum()
print(f"Duplicatas (ra, ano): {dups}")
assert dups == 0, "❌ Duplicatas encontradas na chave (ra, ano)"
print("✅ Chave (ra, ano) única")

# 6.2 — Linhas por ano
print(f"\nLinhas por ano:")
print(alunos["ano"].value_counts().sort_index().to_string())

In [ ]:
# 6.3 — Missing por coluna, pós-limpeza
missing = (alunos.isna().sum() / len(alunos) * 100).round(1)
missing = missing[missing > 0].sort_values(ascending=False)
missing_df = missing.to_frame("pct_missing")
missing_df

**Leitura do quadro de missing:**

- `nota_ing` (64%) — missing **estrutural**: só algumas fases têm inglês. Por isso criamos a feature `tem_nota_ingles`.
- `escola` (62%) — só existe em PEDE2024, nos outros anos é `NaN` por design.
- `ipp` (34%) — só existe em 2023 e 2024 (não existe em 2022).
- Outros (5-8%) — missing aleatório proporcional aos alunos novos/ingressantes sem avaliação completa.

Nenhum desses missings exige imputação agora — eles são **informativos** e serão tratados na modelagem.

In [ ]:
# 6.4 — Distribuição de Fase
print("Distribuição de Fase canônica (0-9):")
fase_dist = alunos.groupby(["ano", "fase"]).size().unstack(fill_value=0)
fase_dist

In [ ]:
# 6.5 — Distribuição de Pedra por ano
pedra_dist = alunos.groupby(["ano", "pedra"]).size().unstack(fill_value=0)
print("Distribuição de Pedra por ano:")
pedra_dist

## 7. Visualizações exploratórias

### 7.1 Distribuição dos 7 indicadores por ano

Boxplots pra cada indicador, comparando 2022, 2023 e 2024.

In [ ]:
indicadores = ["inde", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, ind in zip(axes.flat, indicadores):
    dados = alunos[["ano", ind]].dropna()
    sns.boxplot(data=dados, x="ano", y=ind, ax=ax, width=0.5)
    ax.set_title(ind.upper(), fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_ylim(-0.5, 10.5)

plt.suptitle("Distribuição dos indicadores por ano", fontsize=14, y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

**Leitura:**

- **INDE** sobe consistentemente (7.04 → 7.34 → 7.40) — programa funcionando ✅
- **IAN** é praticamente categórico (valores 2.5, 5.0, 10.0) por construção da fórmula — faz sentido, é defasagem discreta
- **IEG** caiu forte em 2024 (mediana 9.0 → 8.33) — **ponto de atenção 🚨**
- **IPS** em 2023 está muito abaixo de 2022 e 2024 — **anomalia 🚨**
- **IPP** só existe a partir de 2023 (boxplot vazio em 2022 é esperado)

### 7.2 Evolução da média do INDE por fase e ano

Agora olhamos a efetividade do programa: dentro de cada fase, o INDE médio melhora ano a ano?

In [ ]:
evolucao = alunos.groupby(["ano", "fase"])["inde"].mean().unstack(level=0)

fig, ax = plt.subplots(figsize=(11, 5))
evolucao.plot(marker="o", ax=ax, linewidth=2)
ax.set_title("INDE médio por fase, ao longo dos anos", fontweight="bold")
ax.set_xlabel("Fase")
ax.set_ylabel("INDE médio")
ax.set_ylim(5, 9)
ax.legend(title="Ano")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3 Matriz de correlação dos indicadores

Uma das perguntas de negócio é se IEG tem relação direta com IDA e IPV — a matriz abaixo responde de primeira vista.

In [ ]:
corr_cols = ["inde", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv"]
corr = alunos[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Correlação entre indicadores (todos os anos)", fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

**Insights imediatos:**

- **INDE correlaciona forte com IDA e IPV** (esperado — são componentes pesados na fórmula)
- **IEG ↔ IPV** é alta → confirma a pergunta de negócio #3 (engajamento prediz ponto de virada)
- **IAN é o mais descolado** (quase ortogonal) — é o indicador mais "estrutural", depende da idade/série não do comportamento

### 7.4 INDE por Pedra (sanity check)

A pedra é atribuída a partir de faixas fixas de INDE. Os boxplots devem ser basicamente degraus sem sobreposição — esse é o "sanity check" de que a base está consistente.

In [ ]:
ordem_pedras = ["Quartzo", "Ágata", "Ametista", "Topázio"]
dados_pedra = alunos.dropna(subset=["pedra", "inde"])
dados_pedra = dados_pedra[dados_pedra["pedra"].isin(ordem_pedras)]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=dados_pedra,
    x="pedra", y="inde",
    order=ordem_pedras,
    palette=["#e63946", "#f4a261", "#2a9d8f", "#264653"],
    ax=ax,
)
# Linhas de corte oficiais do PEDE
for y, label in [(5.506, "Q→Á"), (6.868, "Á→Am"), (8.230, "Am→T")]:
    ax.axhline(y, color="gray", linestyle="--", alpha=0.5)
    ax.text(3.4, y, label, fontsize=8, color="gray", va="center")

ax.set_title("INDE por Pedra — os cortes oficiais batem com os dados?", fontweight="bold")
ax.set_xlabel("Pedra")
ax.set_ylabel("INDE")
plt.tight_layout()
plt.show()

## 8. Investigação dos "mistérios"

Três padrões anômalos detectados no diagnóstico merecem investigação:

1. **IPS em 2023** — mediana cai de 7.5 (2022) para 5.0 (2023) e volta pra 7.5 (2024). Mudança metodológica?
2. **IEG em 2024** — queda forte (mediana 9.0 → 8.33)
3. **"INCLUIR" em 2024** — 38 alunos com pedra fantasma

### 8.1 Mistério IPS 2023

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, ano in zip(axes, [2022, 2023, 2024]):
    sns.histplot(
        alunos[alunos["ano"] == ano]["ips"].dropna(),
        bins=30, ax=ax, color=sns.color_palette("Set2")[ano - 2022],
    )
    ax.set_title(f"IPS — {ano}", fontweight="bold")
    ax.set_xlabel("IPS")
    ax.set_xlim(0, 10)

plt.suptitle("Distribuição do IPS ano a ano", fontsize=13, y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nPercentis do IPS:")
for ano in [2022, 2023, 2024]:
    s = alunos[alunos["ano"] == ano]["ips"].dropna()
    print(f"  {ano}: mediana={s.median():.2f}, p25={s.quantile(0.25):.2f}, p75={s.quantile(0.75):.2f}, n={len(s)}")

**Hipótese:** em 2023 o IPS parece **bimodal** — uma concentração grande perto de 2.5 (piso) e outra em 7.5. Isso geralmente indica **mudança metodológica** (nova equipe de psicologia, nova escala, novo critério). Em 2024 volta ao padrão de 2022.

**Decisão:** não tocar no dado. Registramos o achado como **insight livre** pra apresentação executiva e evitamos tirar conclusões causais a partir de comparações IPS 2022 vs 2023. Se virar feature do modelo, usamos IPS normalizado dentro do ano (z-score), nunca valor absoluto entre anos.

### 8.2 Mistério IEG 2024

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for ano in [2022, 2023, 2024]:
    sns.kdeplot(
        alunos[alunos["ano"] == ano]["ieg"].dropna(),
        ax=ax, label=str(ano), linewidth=2.5, fill=True, alpha=0.15,
    )
ax.set_title("Distribuição do IEG ano a ano", fontweight="bold")
ax.set_xlabel("IEG")
ax.set_xlim(0, 10)
ax.legend(title="Ano")
plt.tight_layout()
plt.show()

# IEG por fase em 2024 vs 2023
ieg_fase = alunos[alunos["ano"].isin([2023, 2024])].groupby(["ano", "fase"])["ieg"].mean().unstack("ano")
print("\nIEG médio por fase — 2023 vs 2024:")
print(ieg_fase.round(2))
print("\nDelta 2024 - 2023 por fase:")
print((ieg_fase[2024] - ieg_fase[2023]).round(2).to_string())

**Achado:** a queda de IEG em 2024 é **transversal** — acontece em quase todas as fases, não é um problema de uma turma específica. Isso pode indicar algo sistêmico: fadiga pós-pandemia persistente, mudança no método de avaliação de engajamento, ou aumento do tamanho do programa diluindo a atenção por aluno (1156 alunos em 2024 vs 1014 em 2023 = +14%).

Outro **insight forte pra apresentação executiva.** 

### 8.3 Os 102 alunos sem pedra em 2024 — quem são?

In [ ]:
sem_pedra_mask = (alunos["ano"] == 2024) & (alunos["pedra"].isna())
sem_pedra = alunos[sem_pedra_mask].copy()

print(f"Alunos sem pedra em 2024: {len(sem_pedra)}")
print(f"\nDistribuição por fase:")
print(sem_pedra["fase"].value_counts().sort_index())
print(f"\nIdade por fase:")
print(sem_pedra.groupby("fase")["idade"].describe()[["count", "mean", "min", "max"]].round(1))
print(f"\nAno de ingresso no programa:")
print(sem_pedra["ano_ingresso"].value_counts().sort_index())
print(f"\nQuantos têm INDE preenchido? {sem_pedra['inde'].notna().sum()}")

**Descoberta:** os 102 alunos sem pedra em 2024 são **todos do ensino superior**:

- **64 alunos na fase 8** (1º–3º ano universitário, idade média 20)
- **38 alunos na fase 9** (universitários pós-fase-8, idade média 22, rotulados "INCLUIR" no xlsx)

E o mais revelador: **86 deles ingressaram em 2021**. Não são "novos ingressantes" do programa — são **alunos antigos que envelheceram e saíram do alcance do sistema tradicional de pedras do PEDE**, que foi desenhado pra fundamental e médio.

**O que isso diz sobre a Passos Mágicos:** a ONG tá num momento de transição estrutural — começou focada em crianças/adolescentes e agora tem que criar ferramentas de avaliação para o ensino superior. O PEDE original não contempla esses alunos. Isso é um **insight de negócio de alto valor** pra apresentação executiva: *"O programa amadureceu com seus alunos. É hora do PEDE amadurecer junto."*

**Impacto no modelo preditivo:** esses 102 serão automaticamente excluídos da modelagem porque não têm INDE (nossa variável-base pro target). Mas seguem na análise descritiva — eles são parte da história da ONG.

## 9. Retenção e evasão — base de pares `(t, t+1)`

Essa análise é **central** pra viabilidade do modelo forward-looking: precisamos entender quantos pares `(ano t, ano t+1)` temos disponíveis.

In [ ]:
ra_por_ano = {ano: set(alunos[alunos["ano"] == ano]["ra"]) for ano in [2022, 2023, 2024]}

print("Retenção ano-a-ano:")
print(f"  2022 → 2023: {len(ra_por_ano[2022] & ra_por_ano[2023]):4d} de {len(ra_por_ano[2022])} "
      f"({len(ra_por_ano[2022] & ra_por_ano[2023]) / len(ra_por_ano[2022]) * 100:.1f}%)")
print(f"  2023 → 2024: {len(ra_por_ano[2023] & ra_por_ano[2024]):4d} de {len(ra_por_ano[2023])} "
      f"({len(ra_por_ano[2023] & ra_por_ano[2024]) / len(ra_por_ano[2023]) * 100:.1f}%)")
print(f"\nPresentes nos 3 anos: {len(ra_por_ano[2022] & ra_por_ano[2023] & ra_por_ano[2024])}")
print(f"\nApenas 2022: {len(ra_por_ano[2022] - ra_por_ano[2023] - ra_por_ano[2024])}")
print(f"Apenas 2023: {len(ra_por_ano[2023] - ra_por_ano[2022] - ra_por_ano[2024])}")
print(f"Apenas 2024: {len(ra_por_ano[2024] - ra_por_ano[2022] - ra_por_ano[2023])}")

In [ ]:
# Diagrama de fluxo
categorias = {
    "Presente nos 3 anos": len(ra_por_ano[2022] & ra_por_ano[2023] & ra_por_ano[2024]),
    "2022 + 2023 apenas": len((ra_por_ano[2022] & ra_por_ano[2023]) - ra_por_ano[2024]),
    "2023 + 2024 apenas": len((ra_por_ano[2023] & ra_por_ano[2024]) - ra_por_ano[2022]),
    "2022 + 2024 apenas": len((ra_por_ano[2022] & ra_por_ano[2024]) - ra_por_ano[2023]),
    "Apenas 2022 (evasão)": len(ra_por_ano[2022] - ra_por_ano[2023] - ra_por_ano[2024]),
    "Apenas 2023 (transit.)": len(ra_por_ano[2023] - ra_por_ano[2022] - ra_por_ano[2024]),
    "Apenas 2024 (novos)": len(ra_por_ano[2024] - ra_por_ano[2022] - ra_por_ano[2023]),
}

fig, ax = plt.subplots(figsize=(11, 5))
cores = sns.color_palette("Set2", len(categorias))
bars = ax.barh(list(categorias.keys()), list(categorias.values()), color=cores)
ax.set_title("Fluxo de alunos entre os 3 anos", fontweight="bold")
ax.set_xlabel("Nº de alunos (RAs únicos)")
for bar, valor in zip(bars, categorias.values()):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2, str(valor), va="center")
plt.tight_layout()
plt.show()

**Implicações diretas pro modelo:**

- **Treino (pares 2022→2023):** 600 alunos
- **Teste (pares 2023→2024):** 765 alunos
- **Total:** 1365 pares aluno-ano pra modelagem

Dataset é pequeno, mas suficiente pra GBMs tabulares. **E os ~250 alunos que evadiram entre 2022-2023 são ouro pra análise descritiva** — uma das perguntas "insights livres" da Etapa 3 vai investigar *quem evade* e se o modelo conseguiria ter previsto a evasão.

## 10. Export

Salvamos o dataset limpo em `data/processed/alunos_long.parquet`. Todos os notebooks seguintes vão ler *deste* arquivo, não do xlsx original.

In [ ]:
OUTPUT_PATH = DATA_PROCESSED / "alunos_long.parquet"
alunos.to_parquet(OUTPUT_PATH, index=False)

print(f"✅ Salvo em: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"   Shape:   {alunos.shape}")
print(f"   Tamanho: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

# Sanity read-back
rb = pd.read_parquet(OUTPUT_PATH)
assert rb.shape == alunos.shape
print(f"\n✅ Read-back OK — arquivo legível e íntegro")

## ✅ Resumo da Etapa 2

### Dataset consolidado
- **3030 linhas × 27 colunas** (860 + 1014 + 1156, um por aluno-ano)
- Schema canônico com nomes em `snake_case`
- Chave `(ra, ano)` garantidamente única
- Exportado em `data/processed/alunos_long.parquet`

### Principais achados
1. **Schema inconsistente** entre anos — resolvido com funções de limpeza por aba
2. **IPP não existe em 2022** — limita features temporais desse indicador a 2023→2024
3. **Inglês 64% missing estrutural** — criada feature `tem_nota_ingles`
4. **Idade 2023 com bug datetime** — corrigido via `.day`
5. **38 universitários ingressantes sem pedra em 2024** — excluídos do modelo, preservados no dataset
6. **IPS 2023 anômalo (bimodal, mediana 5)** — provável mudança metodológica. Insight livre.
7. **IEG em queda em 2024** — transversal entre fases, insight livre forte
8. **Retenção ano-a-ano ~70-75%** — base de 1365 pares (t, t+1) pra modelagem

### Próximo passo
**Etapa 3 — Análise das 11 perguntas de negócio** (`02_perguntas_negocio.ipynb`)